In [61]:
import os
import pandas as pd
from dotenv import load_dotenv
from gsheet_loader import load_sheet

load_dotenv()

# Load a Google Sheet URL from .env
url = os.getenv("GOOGLE_SHEET_URL")
if not url:
    raise ValueError("GOOGLE_SHEET_URL is not set in .env")

data = load_sheet(url)

In [62]:
df = pd.read_csv(data)

In [63]:
df.drop('submittedAt', axis=1, inplace=True)

In [64]:
df['B5_time_with_parents_hours'] = df['B5_time_with_parents_hours'].str.replace('â', '-', regex=False)
df['D1_daily_screen'] = df['D1_daily_screen'].str.replace('â', '-', regex=False)
df['D2_weekend_screen'] = df['D2_weekend_screen'].str.replace('â', '-', regex=False)
df['D5_parents_screen_time'] = df['D5_parents_screen_time'].str.replace('â', '-', regex=False)
df['E1_sleep_duration'] = df['E1_sleep_duration'].str.replace('â', '-', regex=False)

## **Outlier Detection**

In [65]:
# Inspect raw values to diagnose parsing issue
print("Unique values in B5_time_with_parents_hours:")
print(df['B5_time_with_parents_hours'].unique())

print("\nSample repr (shows hidden characters):")
for v in df['B5_time_with_parents_hours'].dropna().unique()[:10]:
    print(repr(v))


Unique values in B5_time_with_parents_hours:
['2-4 hours' '4-6 hours' '1-2 hours' 'More than 6 hours'
 'Less than 1 hour']

Sample repr (shows hidden characters):
'2-4 hours'
'4-6 hours'
'1-2 hours'
'More than 6 hours'
'Less than 1 hour'


In [66]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages
from matplotlib import gridspec

all_cols = list(df.columns)
numeric_cols = []
categorical_cols = []

for col in all_cols:
    converted = pd.to_numeric(df[col], errors='coerce')
    if converted.notna().sum() >= len(df) * 0.5:
        numeric_cols.append(col)
    else:
        categorical_cols.append(col)

# ── Compute summaries ────────────────────────────────────────────────────────
num_summary = []
for col in numeric_cols:
    series = pd.to_numeric(df[col], errors='coerce').dropna()
    Q1, Q3 = series.quantile(0.25), series.quantile(0.75)
    IQR = Q3 - Q1
    iqr_out = series[(series < Q1 - 1.5 * IQR) | (series > Q3 + 1.5 * IQR)]
    z_scores = (series - series.mean()) / series.std()
    z_out = series[z_scores.abs() > 3]
    num_summary.append({
        'Column': col, 'N': len(series),
        'Mean': round(series.mean(), 2), 'Std': round(series.std(), 2),
        'IQR Outliers': len(iqr_out), 'Z-Score Outliers': len(z_out)
    })

cat_summary = []
for col in categorical_cols:
    counts = df[col].value_counts(dropna=True)
    freq = counts.values.astype(float)
    Q1, Q3 = np.percentile(freq, 25), np.percentile(freq, 75)
    IQR = Q3 - Q1
    iqr_mask = (freq < Q1 - 1.5 * IQR) | (freq > Q3 + 1.5 * IQR)
    iqr_out_cats = counts.index[iqr_mask].tolist()
    mean, std = freq.mean(), freq.std()
    z_scores = (freq - mean) / std if std > 0 else np.zeros(len(freq))
    z_mask = np.abs(z_scores) > 3
    z_out_cats = counts.index[z_mask].tolist()
    cat_summary.append({
        'Column': col,
        'Unique Values': len(counts),
        'Mean Freq': round(mean, 1),
        'Std Freq': round(std, 1),
        'IQR Outlier Categories': len(iqr_out_cats),
        'Z-Score Outlier Categories': len(z_out_cats),
        'IQR Outlier Values': ', '.join(str(v) for v in iqr_out_cats[:3]) + ('...' if len(iqr_out_cats) > 3 else ''),
        'Z-Score Outlier Values': ', '.join(str(v) for v in z_out_cats[:3]) + ('...' if len(z_out_cats) > 3 else '')
    })

num_df = pd.DataFrame(num_summary)
cat_df = pd.DataFrame(cat_summary)

# ── PDF ──────────────────────────────────────────────────────────────────────
pdf_path = 'outlier_detection_report.pdf'

def draw_table(ax, dataframe, title):
    ax.axis('off')
    ax.set_title(title, fontsize=11, fontweight='bold', pad=10)
    tbl = ax.table(
        cellText=dataframe.values,
        colLabels=dataframe.columns,
        cellLoc='center', loc='center'
    )
    tbl.auto_set_font_size(False)
    tbl.set_fontsize(7)
    tbl.auto_set_column_width(col=list(range(len(dataframe.columns))))
    for (row, col), cell in tbl.get_celld().items():
        cell.set_edgecolor('#cccccc')
        if row == 0:
            cell.set_facecolor('#2c3e50')
            cell.set_text_props(color='white', fontweight='bold')
        else:
            cell.set_facecolor('#f9f9f9' if row % 2 == 0 else 'white')

with PdfPages(pdf_path) as pdf:

    # ── Page 1: Title ────────────────────────────────────────────────────────
    fig = plt.figure(figsize=(8.5, 11))
    fig.patch.set_facecolor('white')
    ax = fig.add_subplot(111)
    ax.axis('off')
    ax.text(0.5, 0.6, 'Outlier Detection Report', ha='center', va='center',
            fontsize=24, fontweight='bold', transform=ax.transAxes)
    ax.text(0.5, 0.5, 'IQR + Z-Score Analysis — All Columns', ha='center', va='center',
            fontsize=14, transform=ax.transAxes, color='gray')
    ax.text(0.5, 0.4, f'Total Rows: {len(df)}   |   Numeric Columns: {len(numeric_cols)}   |   Categorical Columns: {len(categorical_cols)}',
            ha='center', va='center', fontsize=11, transform=ax.transAxes)
    pdf.savefig(fig, facecolor='white')
    plt.close()

    # ── Page 2: Numeric summary table ────────────────────────────────────────
    fig, ax = plt.subplots(figsize=(8.5, 11))
    fig.patch.set_facecolor('white')
    draw_table(ax, num_df, 'Numeric Columns — IQR + Z-Score Outlier Summary')
    plt.tight_layout()
    pdf.savefig(fig, facecolor='white')
    plt.close()

    # ── Page 3: Categorical summary table ────────────────────────────────────
    fig, ax = plt.subplots(figsize=(8.5, 11))
    fig.patch.set_facecolor('white')
    draw_table(ax, cat_df, 'Categorical Columns — IQR + Z-Score on Category Frequencies')
    plt.tight_layout()
    pdf.savefig(fig, facecolor='white')
    plt.close()

    # ── Pages: Numeric boxplots (4 per page) ─────────────────────────────────
    cols_per_page = 4
    for page_start in range(0, len(numeric_cols), cols_per_page):
        batch = numeric_cols[page_start:page_start + cols_per_page]
        fig, axes = plt.subplots(1, cols_per_page, figsize=(8.5, 4))
        fig.patch.set_facecolor('white')
        fig.suptitle('Numeric Columns — IQR Outliers (red dots)', fontsize=11, fontweight='bold')
        axes = axes.flatten()
        for i, col in enumerate(batch):
            series = pd.to_numeric(df[col], errors='coerce').dropna()
            Q1, Q3 = series.quantile(0.25), series.quantile(0.75)
            IQR = Q3 - Q1
            iqr_out = series[(series < Q1 - 1.5 * IQR) | (series > Q3 + 1.5 * IQR)]
            axes[i].boxplot(series, vert=True)
            if len(iqr_out):
                axes[i].scatter([1]*len(iqr_out), iqr_out, color='red', zorder=5, s=20, label=f'{len(iqr_out)} outliers')
                axes[i].legend(fontsize=7)
            axes[i].set_title(col, fontsize=8)
            axes[i].tick_params(labelsize=7)
        for j in range(len(batch), cols_per_page):
            axes[j].set_visible(False)
        plt.tight_layout()
        pdf.savefig(fig, facecolor='white')
        plt.close()

    # ── Pages: Categorical bar charts (3 per page) ───────────────────────────
    cols_per_page = 3
    for page_start in range(0, len(categorical_cols), cols_per_page):
        batch = categorical_cols[page_start:page_start + cols_per_page]
        fig, axes = plt.subplots(1, cols_per_page, figsize=(8.5, 5))
        fig.patch.set_facecolor('white')
        fig.suptitle('Categorical Columns — IQR Outlier Categories in Red', fontsize=11, fontweight='bold')
        axes = axes.flatten()
        for i, col in enumerate(batch):
            counts = df[col].value_counts(dropna=True)
            freq = counts.values.astype(float)
            Q1, Q3 = np.percentile(freq, 25), np.percentile(freq, 75)
            IQR = Q3 - Q1
            iqr_mask = (freq < Q1 - 1.5 * IQR) | (freq > Q3 + 1.5 * IQR)
            colors = ['red' if m else 'steelblue' for m in iqr_mask]
            axes[i].bar(range(len(counts)), freq, color=colors)
            axes[i].set_xticks(range(len(counts)))
            axes[i].set_xticklabels(counts.index, rotation=45, ha='right', fontsize=6)
            axes[i].axhline(y=Q1 - 1.5 * IQR, color='orange', linestyle='--', linewidth=1, label='IQR bounds')
            axes[i].axhline(y=Q3 + 1.5 * IQR, color='orange', linestyle='--', linewidth=1)
            axes[i].set_title(col, fontsize=8)
            axes[i].tick_params(labelsize=7)
            axes[i].legend(fontsize=6)
        for j in range(len(batch), cols_per_page):
            axes[j].set_visible(False)
        plt.tight_layout()
        pdf.savefig(fig, facecolor='white')
        plt.close()

print(f"PDF saved: {pdf_path}")


PDF saved: outlier_detection_report.pdf
